In [5]:
import cv2
import json
import os
import glob

# --- CONFIG ---
DATASET_BASE = "D:/GitHub/BaseballPitch/modules/pitcher_segmentation/finetuning_dataset"
SPLIT = "train"  # "train" or "val" or "test"
PITCH_TYPE = "CH - Changeup"
# Progress tracking file — records which reviews are completed
REVIEW_PROGRESS_FILE = f"{DATASET_BASE}/review_progress.json"
# --------------

CLASS_NAMES = ['pitcher']
CLASS_COLORS = [(0, 255, 0)]  # Green for pitcher

In [ ]:
def _load_review_progress():
    if os.path.exists(REVIEW_PROGRESS_FILE):
        with open(REVIEW_PROGRESS_FILE, "r") as f:
            return json.load(f)
    return {}

def _save_review_progress(progress):
    os.makedirs(os.path.dirname(REVIEW_PROGRESS_FILE), exist_ok=True)
    with open(REVIEW_PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)

def _review_progress_key():
    return f"review/{SPLIT}/{PITCH_TYPE}"

def draw_yolo_box(img, label_line):
    """Draw bounding box from YOLO format label"""
    h, w = img.shape[:2]
    label_line = label_line.replace('\\n', '').strip()
    parts = label_line.strip().split()
    if len(parts) < 5:
        return
    
    # YOLO format: class x_center y_center width height
    cls = int(parts[0])
    x_c, y_c, box_w, box_h = map(float, parts[1:5])
    
    # Convert box to pixel coordinates
    x_c *= w
    y_c *= h
    box_w *= w
    box_h *= h
    
    x1 = int(x_c - box_w/2)
    y1 = int(y_c - box_h/2)
    x2 = int(x_c + box_w/2)
    y2 = int(y_c + box_h/2)
    
    # Get class info
    class_name = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f"Class {cls}"
    color = CLASS_COLORS[cls] if cls < len(CLASS_COLORS) else (255, 255, 255)
    
    # Draw bounding box
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    
    # Draw label background
    label_text = class_name
    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cv2.rectangle(img, (x1, y1 - text_h - 10), (x1 + text_w + 10, y1), color, -1)
    cv2.putText(img, label_text, (x1 + 5, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

def review_labels():
    progress = _load_review_progress()
    progress_key = _review_progress_key()

    # Check if already reviewed
    if progress_key in progress and progress[progress_key].get("status") == "done":
        print(f"⏭️  Already reviewed: {SPLIT}/{PITCH_TYPE}")
        print(f"   (Kept: {progress[progress_key].get('kept', 0)}, Deleted: {progress[progress_key].get('deleted', 0)})")
        print("   Delete the progress entry or change SPLIT/PITCH_TYPE to re-review.")
        return

    img_dir = os.path.join(DATASET_BASE, "images", SPLIT, PITCH_TYPE)
    label_dir = os.path.join(DATASET_BASE, "labels", SPLIT, PITCH_TYPE)
    
    # Get all images
    images = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    
    if not images:
        print(f"No images found in {img_dir}")
        return
    
    initial_count = len(images)
    print(f"Found {len(images)} images in {SPLIT}/{PITCH_TYPE}")
    print("\nControls:")
    print("  SPACE/RIGHT ARROW/S - Next image")
    print("  LEFT ARROW/A - Previous image")
    print("  D - Delete current image and label")
    print("  Q/ESC - Quit")
    print("\n" + "="*50)
    
    idx = 0
    deleted_count = 0
    
    while idx < len(images):
        img_path = images[idx]
        img_name = os.path.basename(img_path)
        label_path = os.path.join(label_dir, img_name.replace('.jpg', '.txt'))
        
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Failed to load {img_path}")
            idx += 1
            continue
        
        # Draw bounding boxes if label exists
        has_label = False
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    draw_yolo_box(img, line)
                has_label = True
        
        # Add info overlay
        info_text = f"[{idx+1}/{len(images)}] {img_name}"
        if not has_label:
            info_text += " (NO LABEL)"
        cv2.putText(img, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(img, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 1)
        
        # Show image
        cv2.imshow("Label Review", img)
        
        # Wait for key (use waitKeyEx for reliable arrow keys on Windows)
        key = cv2.waitKeyEx(0)

        # Quit (partial — don't mark done)
        if key in (ord('q'), ord('Q'), 27):  # Q or ESC
            break
        # Delete current image and label
        elif key in (ord('d'), ord('D')):  # Delete
            # Delete both image and label
            os.remove(img_path)
            if os.path.exists(label_path):
                os.remove(label_path)
            images.pop(idx)
            deleted_count += 1
            print(f"Deleted: {img_name}")
            idx = max(0, idx)  # Stay at same position
        # Next image
        elif key in (
            32,               # Space
            2555904,          # Right arrow (Windows)
            83,               # Right arrow (Linux/X11)
            ord('s'), ord('S')  # 'S' key
        ):
            idx = min(idx + 1, len(images) - 1)
        # Previous image
        elif key in (
            2424832,          # Left arrow (Windows)
            81,               # Left arrow (Linux/X11)
            8,                # Backspace
            ord('a'), ord('A')  # 'A' key
        ):
            idx = max(idx - 1, 0)
    
    cv2.destroyAllWindows()

    # If we reached the end, mark as done
    finished = (idx >= len(images) - 1) or (len(images) == 0)
    kept = len(images)

    progress[progress_key] = {
        "status": "done" if finished else "partial",
        "initial": initial_count,
        "kept": kept,
        "deleted": deleted_count,
    }
    _save_review_progress(progress)

    print(f"\nReview {'complete' if finished else 'paused'}!")
    print(f"  Initial:   {initial_count}")
    print(f"  Deleted:   {deleted_count}")
    print(f"  Remaining: {kept}")

In [12]:
review_labels()

Found 540 images in train/CH - Changeup

Controls:
  SPACE/RIGHT ARROW/S - Next image
  LEFT ARROW/A - Previous image
  D - Delete current image and label
  Q/ESC - Quit


Review paused!
  Initial:   540
  Deleted:   0
  Remaining: 540


In [11]:
# --- Review Progress Summary ---
progress = _load_review_progress()

review_entries = {k: v for k, v in progress.items() if k.startswith("review/")}

if not review_entries:
    print("No review progress recorded yet.")
else:
    done_count = sum(1 for v in review_entries.values() if v.get("status") == "done")
    partial_count = sum(1 for v in review_entries.values() if v.get("status") == "partial")
    total_deleted = sum(v.get("deleted", 0) for v in review_entries.values())
    total_kept = sum(v.get("kept", 0) for v in review_entries.values())

    print(f"{'='*60}")
    print(f"Review Progress Summary")
    print(f"{'='*60}")
    print(f"  Completed:  {done_count}")
    print(f"  Partial:    {partial_count}")
    print(f"  Total kept: {total_kept}")
    print(f"  Total deleted: {total_deleted}")
    print(f"{'='*60}")

    for key in sorted(review_entries):
        entry = review_entries[key]
        status = entry.get("status", "unknown")
        icon = "✅" if status == "done" else "⏸️"
        parts = key.split("/", 2)  # review / split / pitch_type
        split_name = parts[1] if len(parts) > 1 else "?"
        pitch_name = parts[2] if len(parts) > 2 else "?"
        print(f"  {icon} [{split_name}] {pitch_name}  —  "
              f"kept: {entry.get('kept', 0)}, deleted: {entry.get('deleted', 0)}")

No review progress recorded yet.
